# Appendix C1. VSRI: Sensitivity of LSTM Results to Random Initialization

In [ ]:
# --- ENV FIRST ---
import os
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
# --- IMPORTS ---
import random
import numpy as np
import tensorflow as tf
# --- TF CONFIG ---
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.set_visible_devices([], 'GPU')
# --- RESET ---
tf.keras.backend.clear_session()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import recall_score, f1_score, roc_auc_score, confusion_matrix, accuracy_score, roc_curve, precision_score
import itertools
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers
try:
  import keras_tuner as kt
except:
  !pip install keras-tuner
  import keras_tuner as kt
from google.colab import files
from google.colab import drive
import pandas as pd
import io
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 1.2 MB/s eta 0:00:00


In [ ]:
def import_data(file_path):
  try:
    drive.mount('/content/drive', force_remount=True)
    # Check if file exists
    if os.path.exists(file_path):
      df = pd.read_parquet(file_path)
      print(f"Loaded dataframe from Drive ({file_path})")
    else:
      raise FileNotFoundError(f"File not found at {file_path}")

  except Exception as e:
    print(f"Drive not available or file missing: {e}")
    print("Please upload dataframe manually.")
    uploaded = files.upload()

    # Automatically read the uploaded file
    file_name = list(uploaded.keys())[0]  # pick the first uploaded file
    try:
      df = pd.read_parquet(io.BytesIO(uploaded[file_name]))
    except:
      print("Wrong file extension. Parquet file required.")
    print(f"Loaded {file_name} from manual upload.")
    return df

In [ ]:
def train_classifier(model, X_train, y_train, X_test, y_test, X_val=False, y_val=False, threshold=0.5, early_stopping=False):
  is_keras = hasattr(model, "fit") and hasattr(model, "predict") and not hasattr(model, "predict_proba")
  if is_keras:
    if early_stopping:
      model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, class_weight=class_weight, validation_data=(X_val, y_val), shuffle=False, callbacks=[combined_metric, es])
    else:
      model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, class_weight=class_weight, validation_data=(X_val, y_val), shuffle=False)
    y_score = model.predict(X_test).ravel()
    y_pred = (y_score >= threshold).astype(int)
  else:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

  # Output Following Metrics:
  recall = recall_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred)
  roc_auc = roc_auc_score(y_test, y_score)
  cm = confusion_matrix(y_test, y_pred)

  return recall, f1, roc_auc, cm, y_score, y_pred

In [ ]:
file_path = "..."
df_sse = import_data(file_path)

Drive not available or file missing: Error: credential propagation was unsuccessful
Please upload dataframe manually.


Saving df_sse.parquet to df_sse.parquet
Loaded df_sse.parquet from manual upload.


In [ ]:
vsri_lstm = import_data(file_path)

Drive not available or file missing: Error: credential propagation was unsuccessful
Please upload dataframe manually.


Saving vsri_lstm.parquet to vsri_lstm.parquet
Loaded vsri_lstm.parquet from manual upload.


In [ ]:
# build a dataframe with the VSRI_LSTM and the Systemic Stress Events variable (target)
df_model_lstm = pd.concat([vsri_lstm, df_sse['systemic_stress_event']], axis=1, join='inner').dropna()
# split data in training and test set and apply purging
split_date = "2024-03-31"
purge = 20
train = df_model_lstm.loc[:split_date].iloc[:-purge]
test = df_model_lstm[df_model_lstm.index > split_date]
X_train = train[["VSRI_LSTM"]]
y_train = train["systemic_stress_event"]
X_test  = test[["VSRI_LSTM"]]
y_test  = test["systemic_stress_event"]

In [ ]:
timesteps = 60
split_idx = pd.Timestamp("2023-03-01")
X_train_lstm = X_train[X_train.index < split_idx].iloc[:-purge]
y_train_lstm = y_train[y_train.index < split_idx].iloc[:-purge]
X_val_lstm = X_train.loc[split_idx:]
y_val_lstm = y_train.loc[split_idx:]

X_train_values = X_train_lstm.values
y_train_values = y_train_lstm.values
X_test_values = X_test.values
y_test_values = y_test.values
X_val_values = X_val_lstm.values
y_val_values = y_val_lstm.values
X_train_seq = []
y_train_seq = []
X_test_seq = []
y_test_seq = []
X_val_seq = []
y_val_seq = []

for i in range(timesteps, len(X_train_values)):
  X_train_seq.append(X_train_values[i - timesteps:i])
  y_train_seq.append(y_train_values[i])
for i in range(timesteps, len(X_test_values)):
  X_test_seq.append(X_test_values[i - timesteps:i])
  y_test_seq.append(y_test_values[i])
for i in range(timesteps, len(X_val_values)):
  X_val_seq.append(X_val_values[i - timesteps:i])
  y_val_seq.append(y_val_values[i])

X_train_seq = np.array(X_train_seq)
y_train_seq = np.array(y_train_seq)
X_test_seq = np.array(X_test_seq)
y_test_seq = np.array(y_test_seq)
X_val_seq = np.array(X_val_seq)
y_val_seq = np.array(y_val_seq)

In [ ]:
print(X_train_seq.shape,
      y_train_seq.shape,
      X_val_seq.shape,
      y_val_seq.shape,
      X_test_seq.shape,
      y_test_seq.shape)

(1457, 60, 1) (1457,) (192, 60, 1) (192,) (416, 60, 1) (416,)


In [ ]:
class CombinedMetric(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs=None):
    logs = logs or {}
    recall = logs.get("val_recall", 0)
    pr_auc = logs.get("val_pr_auc", 0)
    logs["val_combined"] = 0.3 * recall + 0.7 * pr_auc

In [ ]:
classes = np.unique(y_train_seq)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_seq)
class_weight = dict(zip(classes, weights))
epochs=200
batch_size=32

In [ ]:
seeds = [38, 60, 96, 146, 162]
robustness_results = []
input_shape = X_train_seq.shape[1:]

In [ ]:
for seed in seeds:
  print(f"\nRunning seed {seed}")
  tf.keras.backend.clear_session()
  random.seed(seed)
  np.random.seed(seed)
  tf.keras.utils.set_random_seed(seed)

  # LSTM MODEL
  lstm_model = keras.Sequential()
  lstm_model.add(layers.LSTM(128, return_sequences=True, input_shape=input_shape))
  lstm_model.add(layers.Dropout(0.5))
  lstm_model.add(layers.LSTM(128, return_sequences=True))
  lstm_model.add(layers.LSTM(16, return_sequences=True))
  lstm_model.add(layers.LSTM(8, return_sequences=True))
  lstm_model.add(layers.Dropout(0.3))
  lstm_model.add(layers.LSTM(8, return_sequences=False))
  lstm_model.add(layers.Dropout(0.3))
  # Dense layers
  lstm_model.add(layers.Dense(16, activation="relu"))
  lstm_model.add(layers.Dense(1, activation="sigmoid"))
  # Compile
  lstm_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
                     loss="binary_crossentropy",
                     metrics=[keras.metrics.AUC(name="pr_auc", curve="PR"),
                              keras.metrics.AUC(name="roc_auc", curve="ROC"),
                              tf.keras.metrics.Recall(name="recall"),
                              tf.keras.metrics.Precision(name="precision")])

  combined_metric = CombinedMetric()
  es = tf.keras.callbacks.EarlyStopping(monitor="val_combined",
                                        mode="max",
                                        patience=50,
                                        restore_best_weights=True,
                                        verbose=1)

  # TRAIN
  lstm_results = train_classifier(lstm_model, X_train_seq, y_train_seq, X_test_seq, y_test_seq, X_val=X_val_seq, y_val=y_val_seq, early_stopping=True)
  (lstm_recall, lstm_f1, lstm_roc_auc, lstm_cm, lstm_probs, lstm_preds) = lstm_results
  # Store results
  robustness_results.append({"seed": seed,
                             "recall": lstm_recall,
                             "f1": lstm_f1,
                             "roc_auc": lstm_roc_auc,
                             "tn": lstm_cm[0,0],
                             "fp": lstm_cm[0,1],
                             "fn": lstm_cm[1,0],
                             "tp": lstm_cm[1,1]})


Running seed 38


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 23s 253ms/step - loss: 0.6935 - pr_auc: 0.0731 - precision: 0.0556 - recall: 0.4483 - roc_auc: 0.4836 - val_loss: 0.6930 - val_pr_auc: 0.0521 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000 - val_combined: 0.0365
Epoch 2/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 11s 245ms/step - loss: 0.6934 - pr_auc: 0.0782 - precision: 0.0742 - recall: 0.6983 - roc_auc: 0.4976 - val_loss: 0.6940 - val_pr_auc: 0.0521 - val_precision: 0.0521 - val_recall: 1.0000 - val_roc_auc: 0.5000 - val_combined: 0.3365
Epoch 3/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 11s 247ms/step - loss: 0.6934 - pr_auc: 0.0785 - precision: 0.0729 - recall: 0.7759 - roc_auc: 0.4985 - val_loss: 0.6935 - val_pr_auc: 0.0521 - val_precision: 0.0524 - val_recall: 1.0000 - val_roc_auc: 0.5000 - val_combined: 0.3365
Epoch 4/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 11s 244ms/step - loss: 0.6932 - pr_auc: 0.0796 - precision: 0.0648 - recall: 0.5603 - roc_auc: 0.5000 - val_loss: 0.6941 - val_pr_auc: 0.052

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 22s 245ms/step - loss: 0.6937 - pr_auc: 0.0687 - precision: 0.0000e+00 - recall: 0.0000e+00 - roc_auc: 0.4349 - val_loss: 0.6899 - val_pr_auc: 0.0532 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5110 - val_combined: 0.0372
Epoch 2/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 202ms/step - loss: 0.6930 - pr_auc: 0.0772 - precision: 0.0323 - recall: 0.0086 - roc_auc: 0.4946 - val_loss: 0.6901 - val_pr_auc: 0.0541 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5192 - val_combined: 0.0378
Epoch 3/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 201ms/step - loss: 0.6932 - pr_auc: 0.0810 - precision: 0.0690 - recall: 0.0345 - roc_auc: 0.5037 - val_loss: 0.6903 - val_pr_auc: 0.0538 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5165 - val_combined: 0.0376
Epoch 4/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 215ms/step - loss: 0.6932 - pr_auc: 0.0795 - precision: 0.0980 - recall: 0.0431 - roc_auc: 0.4980 - val_loss: 0.69

...

Epoch 1/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 22s 251ms/step - loss: 0.6937 - pr_auc: 0.0690 - precision: 0.0698 - recall: 0.0517 - roc_auc: 0.4501 - val_loss: 0.6869 - val_pr_auc: 0.0521 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000 - val_combined: 0.0365
Epoch 2/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 221ms/step - loss: 0.6934 - pr_auc: 0.0784 - precision: 0.0617 - recall: 0.0431 - roc_auc: 0.4866 - val_loss: 0.6875 - val_pr_auc: 0.0369 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.3599 - val_combined: 0.0258
Epoch 3/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 219ms/step - loss: 0.6923 - pr_auc: 0.0923 - precision: 0.0870 - recall: 0.0690 - roc_auc: 0.5528 - val_loss: 0.6861 - val_pr_auc: 0.0541 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5192 - val_combined: 0.0378
Epoch 4/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 193ms/step - loss: 0.6934 - pr_auc: 0.0812 - precision: 0.0943 - recall: 0.0862 - roc_auc: 0.5071 - val_loss: 0.6854 - va

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 21s 244ms/step - loss: 0.6934 - pr_auc: 0.0800 - precision: 0.0408 - recall: 0.0690 - roc_auc: 0.5046 - val_loss: 0.6917 - val_pr_auc: 0.0521 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000 - val_combined: 0.0365
Epoch 2/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 213ms/step - loss: 0.6931 - pr_auc: 0.0799 - precision: 0.0345 - recall: 0.0172 - roc_auc: 0.5021 - val_loss: 0.6917 - val_pr_auc: 0.0521 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000 - val_combined: 0.0365
Epoch 3/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 194ms/step - loss: 0.6933 - pr_auc: 0.0798 - precision: 0.0444 - recall: 0.0862 - roc_auc: 0.5180 - val_loss: 0.6911 - val_pr_auc: 0.0521 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000 - val_combined: 0.0365
Epoch 4/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 215ms/step - loss: 0.6932 - pr_auc: 0.0835 - precision: 0.0270 - recall: 0.0086 - roc_auc: 0.5267 - val_loss: 0.6909 - va

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 22s 238ms/step - loss: 0.6936 - pr_auc: 0.0696 - precision: 0.0612 - recall: 0.4310 - roc_auc: 0.4664 - val_loss: 0.6958 - val_pr_auc: 0.0521 - val_precision: 0.0521 - val_recall: 1.0000 - val_roc_auc: 0.5000 - val_combined: 0.3365
Epoch 2/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 217ms/step - loss: 0.6935 - pr_auc: 0.0685 - precision: 0.0806 - recall: 0.6293 - roc_auc: 0.4621 - val_loss: 0.6930 - val_pr_auc: 0.0521 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000 - val_combined: 0.0365
Epoch 3/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 226ms/step - loss: 0.6931 - pr_auc: 0.0777 - precision: 0.0790 - recall: 0.3793 - roc_auc: 0.5080 - val_loss: 0.6925 - val_pr_auc: 0.0521 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000 - val_combined: 0.0365
Epoch 4/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 221ms/step - loss: 0.6931 - pr_auc: 0.0806 - precision: 0.0752 - recall: 0.2672 - roc_auc: 0.5154 - val_loss: 0.6927 - val_pr_au

In [ ]:
df_robustness = pd.DataFrame(robustness_results)
df_robustness = pd.concat([df_robustness, pd.DataFrame({'seed': 23,
                                                        'recall': 0.75,
                                                        'f1': 0.18,
                                                        'roc_auc': 0.77,
                                                        'tn': 239,
                                                        'fp': 153,
                                                        'fn': 6,
                                                        'tp': 18},
                                                        index=[0])], ignore_index=True) #add results from baseline model
df_robustness

,seed,recall,f1,roc_auc,tn,fp,fn,tp
0,38,1.000,0.109091,0.551339,0,392,0,24
1,60,0.750,0.178218,0.700893,232,160,6,18
2,96,0.750,0.180905,0.729911,235,157,6,18
3,146,0.750,0.167442,0.554953,219,173,6,18
4,162,0.625,0.194805,0.663690,277,115,9,15
5,23,0.750,0.180000,0.770000,239,153,6,18


```python
df_robustness.to_csv("df_robustness.csv", index=True)
files.download("df_robustness.csv")
df_robustness.to_parquet("df_robustness.parquet", index=True)
files.download("df_robustness.parquet")
```

In [ ]:
#df_robustness = import_data(file_path)

In [ ]:
df_robustness = df_robustness.set_index('seed')[['recall',	'f1',	'roc_auc']].sort_index()
df_robustness = df_robustness.rename(columns={'recall':'Recall',	'f1':'F1-Score',	'roc_auc':'ROC-AUC'})

In [ ]:
df_robustness.round(2)

,Recall,F1-Score,ROC-AUC
seed,,,
23,0.75,0.18,0.77
38,1.00,0.11,0.55
60,0.75,0.18,0.70
96,0.75,0.18,0.73
146,0.75,0.17,0.55
162,0.62,0.19,0.66


In [ ]:
df_robustness.mean().round(2).rename('mean')

,mean
Recall,0.77
F1-Score,0.17
ROC-AUC,0.66


In [ ]:
df_robustness.std().round(2).rename('std')

,std
Recall,0.12
F1-Score,0.03
ROC-AUC,0.09
